# ERCOT Load + Weather Data Combiner

This notebook:
1. Loads all 8 yearly ERCOT Native Load Excel files (2018–2025)
2. Fixes two known quirks in the raw data
3. Stacks them into one combined load DataFrame
4. Loads the weather CSV you downloaded earlier
5. Merges everything into one final dataset ready for modeling

### Known quirks we handle:
- **Column name inconsistency**: 2018–2020 use `HourEnding`, 2021–2025 use `Hour Ending`
- **24:00 timestamps**: ERCOT marks end-of-day as `12/31/2018 24:00` — pandas can't parse this, so we convert it to the next day at `00:00`

## Step 1: Imports

In [7]:
%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import os

Libraries loaded!


## Step 2: Define File Paths

Update `DATA_FOLDER` to wherever you saved the ERCOT Excel files.
Update `WEATHER_FILE` to wherever you saved the weather CSV from the download notebook.

In [ ]:
# ── UPDATE THESE PATHS ──────────────────────────────────────────────────────
DATA_FOLDER  = "/Users/bestek/PycharmProjects/ercot-electricity-demand-forecasting/raw_data"   # folder containing the ERCOT Excel files
WEATHER_FILE = "/Users/bestek/PycharmProjects/ercot-electricity-demand-forecasting/processed_data/ercot_weather_wide_2018_2025.csv"  # from the weather download notebook
# ────────────────────────────────────────────────────────────────────────────

LOAD_FILES = {
    2018: os.path.join(DATA_FOLDER, "Native_Load_2018.xlsx"),
    2019: os.path.join(DATA_FOLDER, "Native_Load_2019.xlsx"),
    2020: os.path.join(DATA_FOLDER, "Native_Load_2020.xlsx"),
    2021: os.path.join(DATA_FOLDER, "Native_Load_2021.xlsx"),
    2022: os.path.join(DATA_FOLDER, "Native_Load_2022.xlsx"),
    2023: os.path.join(DATA_FOLDER, "Native_Load_2023.xlsx"),
    2024: os.path.join(DATA_FOLDER, "Native_Load_2024.xlsx"),
    2025: os.path.join(DATA_FOLDER, "Native_Load_2025.xlsx"),
}

# Verify files exist
for year, path in LOAD_FILES.items():
    status = "✓" if os.path.exists(path) else "✗ NOT FOUND"
    print(f"  {year}: {status}")

  2018: ✓
  2019: ✓
  2020: ✓
  2021: ✓
  2022: ✓
  2023: ✓
  2024: ✓
  2025: ✓


## Step 3: The Datetime Fix

ERCOT uses `24:00` to represent the end of a day (e.g., `01/01/2018 24:00`).
This is technically valid in some energy standards but pandas cannot parse it.

**The fix:** replace `24:00` with `00:00` and add 1 day, so:
- `01/01/2018 24:00` → `01/02/2018 00:00` ✓

This correctly represents midnight at the boundary between two days.

In [ ]:
def fix_hour_ending(raw_string):
    """
    Convert ERCOT HourEnding string to a proper pandas Timestamp.
    Handles two special cases:
      - '24:00'     → end of day, shift to next day 00:00
      - '02:00 DST' → fall-back hour, shift forward 1 hour to 03:00
                      so it doesn't collide with the real 02:00
    """
    s = str(raw_string).strip()
    
    if "24:00" in s:
        s = s.replace("24:00", "00:00")
        return pd.Timestamp(s) + pd.Timedelta(days=1)
    elif "DST" in s:
        # Remove the DST label, parse, then shift forward 1 hour
        # e.g. '11/04/2018 02:00 DST' → 2018-11-04 03:00:00
        s = s.replace(" DST", "")
        return pd.Timestamp(s) + pd.Timedelta(hours=1)
    else:
        return pd.Timestamp(s)

# Quick tests
print(fix_hour_ending("01/01/2018 24:00"))       # → 2018-01-02 00:00:00
print(fix_hour_ending("11/04/2018 02:00 DST"))   # → 2018-11-04 03:00:00  (no duplicate!)
print(fix_hour_ending("01/01/2018 01:00"))        # → 2018-01-01 01:00:00

2018-01-02 00:00:00
2018-01-01 01:00:00


## Step 4: Load & Stack All ERCOT Files

In [8]:
ZONES = ["COAST", "EAST", "FWEST", "NORTH", "NCENT", "SOUTH", "SCENT", "WEST"]

yearly_dfs = []

for year, path in LOAD_FILES.items():
    print(f"Loading {year}...", end=" ")
    
    df = pd.read_excel(path)
    
    # Step 1: Normalize the column name (handles 'HourEnding' vs 'Hour Ending')
    df.columns = [c.strip().replace(" ", "") for c in df.columns]
    # Now it's always 'HourEnding' regardless of year
    
    # Step 2: Fix the datetime
    df["datetime"] = df["HourEnding"].apply(fix_hour_ending)
    
    # Step 3: Keep only what we need
    df = df[["datetime"] + ZONES + ["ERCOT"]]
    
    # Step 4: Rename zone columns to make it clear these are load (MW) values
    rename_map = {zone: f"{zone}_load_mw" for zone in ZONES}
    rename_map["ERCOT"] = "ERCOT_total_load_mw"
    df = df.rename(columns=rename_map)
    
    yearly_dfs.append(df)
    print(f"Done! {len(df)} rows")

# Stack all years into one DataFrame
df_load = pd.concat(yearly_dfs, ignore_index=True)
df_load = df_load.sort_values("datetime").reset_index(drop=True)

print(f"\nCombined load data: {df_load.shape}")
print(f"Date range: {df_load['datetime'].min()} → {df_load['datetime'].max()}")

Loading 2018... 

/var/folders/3z/gkmtk37953l3ph4v50b84b300000gn/T/ipykernel_44995/3615281046.py:13: FutureWarning: Parsed string "11/04/2018 02:00 DST" included an un-recognized timezone "DST". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  return pd.Timestamp(s)


Done! 8760 rows
Loading 2019... 

/var/folders/3z/gkmtk37953l3ph4v50b84b300000gn/T/ipykernel_44995/3615281046.py:13: FutureWarning: Parsed string "11/03/2019 02:00 DST" included an un-recognized timezone "DST". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  return pd.Timestamp(s)


Done! 8760 rows
Loading 2020... 

...

/var/folders/3z/gkmtk37953l3ph4v50b84b300000gn/T/ipykernel_44995/3615281046.py:13: FutureWarning: Parsed string "11/05/2023 02:00 DST" included an un-recognized timezone "DST". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  return pd.Timestamp(s)


Done! 8760 rows
Loading 2024... 

/var/folders/3z/gkmtk37953l3ph4v50b84b300000gn/T/ipykernel_44995/3615281046.py:13: FutureWarning: Parsed string "11/03/2024 02:00 DST" included an un-recognized timezone "DST". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  return pd.Timestamp(s)


Done! 8784 rows
Loading 2025... Done! 8760 rows

Combined load data: (70128, 10)
Date range: 2018-01-01 01:00:00 → 2026-01-01 00:00:00


/var/folders/3z/gkmtk37953l3ph4v50b84b300000gn/T/ipykernel_44995/3615281046.py:13: FutureWarning: Parsed string "11/02/2025 02:00 DST" included an un-recognized timezone "DST". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  return pd.Timestamp(s)


## Step 5: Quick Sanity Check on Load Data

In [9]:
print("=== MISSING VALUES ===")
print(df_load.isnull().sum())

print("\n=== DUPLICATE TIMESTAMPS ===")
dupes = df_load[df_load.duplicated(subset='datetime', keep=False)]
print(f"Duplicate rows: {len(dupes)}")
if len(dupes) > 0:
    print(dupes[["datetime", "ERCOT_total_load_mw"]].head(10))

print("\n=== SAMPLE DATA ===")
df_load.head()

=== MISSING VALUES ===
datetime               0
COAST_load_mw          0
EAST_load_mw           0
FWEST_load_mw          0
NORTH_load_mw          0
NCENT_load_mw          0
SOUTH_load_mw          0
SCENT_load_mw          0
WEST_load_mw           0
ERCOT_total_load_mw    0
dtype: int64

=== DUPLICATE TIMESTAMPS ===
Duplicate rows: 16
                 datetime  ERCOT_total_load_mw
7368  2018-11-04 02:00:00         29382.002535
7369  2018-11-04 02:00:00         28555.931880
16104 2019-11-03 02:00:00         33324.971467
16105 2019-11-03 02:00:00         33309.960879
24840 2020-11-01 02:00:00         30232.558552
24841 2020-11-01 02:00:00         30650.118113
33744 2021-11-07 02:00:00         32661.760943
33745 2021-11-07 02:00:00         33049.815345
42480 2022-11-06 02:00:00         34706.875462
42481 2022-11-06 02:00:00         34067.955296

=== SAMPLE DATA ===


,datetime,COAST_load_mw,EAST_load_mw,FWEST_load_mw,NORTH_load_mw,NCENT_load_mw,SOUTH_load_mw,SCENT_load_mw,WEST_load_mw,ERCOT_total_load_mw
0,2018-01-01 01:00:00,11425.979115,1852.663959,2823.409245,1135.360907,18584.343617,3831.649454,9151.190703,1762.472684,50567.069682
1,2018-01-01 02:00:00,11408.418023,1850.169452,2809.745403,1136.630855,18524.141392,3988.271046,9144.993712,1754.718094,50617.087977
2,2018-01-01 03:00:00,11405.198365,1858.269586,2797.802576,1135.930264,18532.056616,4076.086451,9141.036615,1747.919615,50694.300087
3,2018-01-01 04:00:00,11450.560138,1879.623596,2807.793880,1146.069491,18647.444612,4154.939804,9157.956866,1755.203307,50999.591693
4,2018-01-01 05:00:00,11631.337459,1876.481320,2822.989206,1154.186967,19002.102222,4247.451523,9214.333628,1774.849690,51723.732017


## Step 6: Load Weather Data

In [ ]:
df_weather = pd.read_csv(WEATHER_FILE, parse_dates=["datetime"])

print("Raw datetime sample:", df_weather['datetime'].iloc[0])
print("Raw datetime dtype: ", df_weather['datetime'].dtype)

# utc=True tells pandas to handle any timezone suffix in the string (e.g. '+00:00')
df_weather['datetime'] = pd.to_datetime(df_weather['datetime'], utc=True)

# Convert from UTC to Central Time (ERCOT's timezone)
df_weather['datetime'] = df_weather['datetime'].dt.tz_convert('America/Chicago')

# Strip timezone so it matches ERCOT load data (which has no timezone label)
df_weather['datetime'] = df_weather['datetime'].dt.tz_localize(None)

print(f"\nParsed datetime sample: {df_weather['datetime'].iloc[0]}")
print(f"Parsed datetime dtype:  {df_weather['datetime'].dtype}")
print(f"\nWeather data shape: {df_weather.shape}")
print(f"Date range: {df_weather['datetime'].min()} → {df_weather['datetime'].max()}")
print(f"\nWeather columns: {df_weather.columns.tolist()}")

Raw datetime sample: 2018-01-01 00:00:00-06:00
Raw datetime dtype:  object

Parsed datetime sample: 2018-01-01 00:00:00
Parsed datetime dtype:  datetime64[ns]

Weather data shape: (70128, 39)
Date range: 2018-01-01 00:00:00 → 2025-12-31 23:00:00

Weather columns: ['datetime', 'COAST_temperature_f', 'EAST_temperature_f', 'FWEST_temperature_f', 'NCENT_temperature_f', 'NORTH_temperature_f', 'SCENT_temperature_f', 'SOUTH_temperature_f', 'WEST_temperature_f', 'COAST_relative_humidity_pct', 'EAST_relative_humidity_pct', 'FWEST_relative_humidity_pct', 'NCENT_relative_humidity_pct', 'NORTH_relative_humidity_pct', 'SCENT_relative_humidity_pct', 'SOUTH_relative_humidity_pct', 'WEST_relative_humidity_pct', 'COAST_wind_speed_mph', 'EAST_wind_speed_mph', 'FWEST_wind_speed_mph', 'NCENT_wind_speed_mph', 'NORTH_wind_speed_mph', 'SCENT_wind_speed_mph', 'SOUTH_wind_speed_mph', 'WEST_wind_speed_mph', 'COAST_solar_radiation_wm2', 'EAST_solar_radiation_wm2', 'FWEST_solar_radiation_wm2', 'NCENT_solar_radiat

## Step 7: Merge Load + Weather

We do a **left join** on `datetime` — this keeps all ERCOT rows and attaches weather data where available.
Any ERCOT hours without a matching weather timestamp will show `NaN` in weather columns (we'll check for these).

In [14]:
df_merged = df_load.merge(df_weather, on="datetime", how="left")

print(f"Merged dataset shape: {df_merged.shape}")
print(f"Rows: {len(df_merged):,}")
print(f"Columns: {len(df_merged.columns)}")

# Check how many rows didn't get weather data
missing_weather = df_merged["COAST_temperature_f"].isnull().sum()
print(f"\nRows missing weather data: {missing_weather} ({missing_weather/len(df_merged)*100:.2f}%)")

Merged dataset shape: (70136, 48)
Rows: 70,136
Columns: 48

Rows missing weather data: 9 (0.01%)


## Step 8: Final Column Overview

In [15]:
print("ALL COLUMNS IN FINAL DATASET:")
print()

load_cols    = [c for c in df_merged.columns if 'load' in c]
temp_cols    = [c for c in df_merged.columns if 'temperature' in c]
wind_cols    = [c for c in df_merged.columns if 'wind' in c]
solar_cols   = [c for c in df_merged.columns if 'radiation' in c or 'irradiance' in c]
cal_cols     = [c for c in df_merged.columns if c in ['hour','day_of_week','month','year','is_weekend','is_holiday']]
humidity_cols= [c for c in df_merged.columns if 'humidity' in c]

print(f"📊 Load columns ({len(load_cols)}):    {load_cols}")
print(f"🌡️  Temperature ({len(temp_cols)}):    {temp_cols}")
print(f"💧 Humidity ({len(humidity_cols)}):       {humidity_cols}")
print(f"💨 Wind ({len(wind_cols)}):             {wind_cols}")
print(f"☀️  Solar ({len(solar_cols)}):           {solar_cols}")
print(f"📅 Calendar ({len(cal_cols)}):         {cal_cols}")

ALL COLUMNS IN FINAL DATASET:

📊 Load columns (9):    ['COAST_load_mw', 'EAST_load_mw', 'FWEST_load_mw', 'NORTH_load_mw', 'NCENT_load_mw', 'SOUTH_load_mw', 'SCENT_load_mw', 'WEST_load_mw', 'ERCOT_total_load_mw']
🌡️  Temperature (8):    ['COAST_temperature_f', 'EAST_temperature_f', 'FWEST_temperature_f', 'NCENT_temperature_f', 'NORTH_temperature_f', 'SCENT_temperature_f', 'SOUTH_temperature_f', 'WEST_temperature_f']
💧 Humidity (8):       ['COAST_relative_humidity_pct', 'EAST_relative_humidity_pct', 'FWEST_relative_humidity_pct', 'NCENT_relative_humidity_pct', 'NORTH_relative_humidity_pct', 'SCENT_relative_humidity_pct', 'SOUTH_relative_humidity_pct', 'WEST_relative_humidity_pct']
💨 Wind (8):             ['COAST_wind_speed_mph', 'EAST_wind_speed_mph', 'FWEST_wind_speed_mph', 'NCENT_wind_speed_mph', 'NORTH_wind_speed_mph', 'SCENT_wind_speed_mph', 'SOUTH_wind_speed_mph', 'WEST_wind_speed_mph']
☀️  Solar (8):           ['COAST_solar_radiation_wm2', 'EAST_solar_radiation_wm2', 'FWEST_solar_r

## Step 9: Preview Final Dataset

In [16]:
# Show a few key columns to confirm the merge looks right
preview_cols = [
    "datetime", 
    "ERCOT_total_load_mw",
    "NCENT_load_mw",         # Dallas — largest zone
    "NCENT_temperature_f",   # matching weather
    "NCENT_wind_speed_mph",
    "NCENT_solar_radiation_wm2",
    "hour", "day_of_week", "is_holiday"
]

df_merged[preview_cols].head(24)  # first 24 hours

,datetime,ERCOT_total_load_mw,NCENT_load_mw,NCENT_temperature_f,NCENT_wind_speed_mph,NCENT_solar_radiation_wm2,hour,day_of_week,is_holiday
0,2018-01-01 01:00:00,50567.069682,18584.343617,20.480000,14.085997,0.0,1.0,0.0,1.0
1,2018-01-01 02:00:00,50617.087977,18524.141392,19.850000,13.350974,0.0,2.0,0.0,1.0
2,2018-01-01 03:00:00,50694.300087,18532.056616,19.220001,13.425728,0.0,3.0,0.0,1.0
3,2018-01-01 04:00:00,50999.591693,18647.444612,18.500000,12.432971,0.0,4.0,0.0,1.0
4,2018-01-01 05:00:00,51723.732017,19002.102222,18.230000,12.432971,0.0,5.0,0.0,1.0
5,2018-01-01 06:00:00,52955.634255,19477.360921,16.340000,11.548207,0.0,6.0,0.0,1.0
6,2018-01-01 07:00:00,54303.082184,19984.222251,16.160000,10.885712,0.0,7.0,0.0,1.0
7,2018-01-01 08:00:00,55099.267435,20289.365819,16.430000,11.340508,10.0,8.0,0.0,1.0
8,2018-01-01 09:00:00,55553.887208,20338.611561,18.320000,12.384578,138.0,9.0,0.0,1.0
9,2018-01-01 10:00:00,56070.390455,20250.287356,20.480000,11.965337,315.0,10.0,0.0,1.0


## Step 10: Create Train / Validate / Test Splits

Based on our project design:
- **Train**: 2018–2023
- **Validate**: 2024 (tune model hyperparameters here)
- **Test**: 2025 (final evaluation — compare to actual data)

In [17]:
train = df_merged[df_merged["datetime"].dt.year <= 2023].copy()
val   = df_merged[df_merged["datetime"].dt.year == 2024].copy()
test  = df_merged[df_merged["datetime"].dt.year == 2025].copy()

print("SPLIT SUMMARY")
print(f"  Train (2018–2023): {len(train):>7,} rows  |  {train['datetime'].min().date()} → {train['datetime'].max().date()}")
print(f"  Validate (2024):   {len(val):>7,} rows  |  {val['datetime'].min().date()} → {val['datetime'].max().date()}")
print(f"  Test (2025):       {len(test):>7,} rows  |  {test['datetime'].min().date()} → {test['datetime'].max().date()}")
print(f"  Total:             {len(df_merged):>7,} rows")

SPLIT SUMMARY
  Train (2018–2023):  52,589 rows  |  2018-01-01 → 2023-12-31
  Validate (2024):     8,785 rows  |  2024-01-01 → 2024-12-31
  Test (2025):         8,761 rows  |  2025-01-01 → 2025-12-31
  Total:              70,136 rows


## Step 11: Save Everything

In [18]:
# Full merged dataset
df_merged.to_csv("ercot_full_dataset_2018_2025.csv", index=False)
print(f"Saved: ercot_full_dataset_2018_2025.csv  ({df_merged.shape[0]:,} rows x {df_merged.shape[1]} cols)")

# Individual splits
train.to_csv("ercot_train_2018_2023.csv", index=False)
val.to_csv("ercot_validate_2024.csv", index=False)
test.to_csv("ercot_test_2025.csv", index=False)

print(f"Saved: ercot_train_2018_2023.csv         ({len(train):,} rows)")
print(f"Saved: ercot_validate_2024.csv           ({len(val):,} rows)")
print(f"Saved: ercot_test_2025.csv               ({len(test):,} rows)")
print("\nAll files saved! Ready for EDA and modeling.")

Saved: ercot_full_dataset_2018_2025.csv  (70,136 rows x 48 cols)
Saved: ercot_train_2018_2023.csv         (52,589 rows)
Saved: ercot_validate_2024.csv           (8,785 rows)
Saved: ercot_test_2025.csv               (8,761 rows)

All files saved! Ready for EDA and modeling.
